In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV, KFold, cross_validate
from sklearn.metrics import make_scorer, r2_score, mean_squared_error, mean_absolute_error

from catboost import CatBoostRegressor
from sklearn.multioutput import MultiOutputRegressor
from scipy.stats import randint, uniform

import os
import joblib

In [2]:
# 1- xgboost
# 2- adaboost
# 3- gradient boosting
# 4- catboost
# 5- lightboost
# 6- random forest
# 7- stacking 01
# 8- stacking 02

In [3]:
df = pd.read_csv("../data/preprocessed_dataset.csv")

df.head(2)

,FA (kg/m3),GGBFS (kg/m3),Coarse aggregate (kg/m3),Fine aggregate (kg/m3),Na2SiO3,NaOH,Sio2 (Dry),Concentration (M) NaOH,Superplasticizer,Total water,Initial curing time (day),Initial curing temp (C),Initial curing rest time (day),Final curing temp (C),Water_to_Binder,Water_to_Activator,SP_ratio,total_material_mass,Total_CO2_kg,Compressive_strength_MPa
0,0.000000,0.069565,-0.127973,-0.495765,0.700467,0.741549,0.756779,0.0,0.095238,0.651360,0.0,0.0,0.0,0.0,0.629716,-0.436364,0.107509,0.885142,512.210033,46.57
1,0.140762,0.162319,-0.465568,0.198785,0.900600,0.950927,0.972882,0.0,0.142857,0.849623,0.0,0.0,0.0,0.0,0.629716,-0.436364,0.159822,0.948367,534.887466,46.01


In [4]:
labels = df[['Total_CO2_kg', 'Compressive_strength_MPa']]
df.drop(columns=['Total_CO2_kg', 'Compressive_strength_MPa'], inplace=True)

In [5]:
X_train, X_test, y_train, y_test = train_test_split(df, labels, random_state=42, test_size=0.20)

# CatBoost

In [6]:
cbr = CatBoostRegressor(
    loss_function="RMSE",
    random_seed=42,
    verbose=0,
    allow_writing_files=False,
    thread_count=1
)

model = MultiOutputRegressor(cbr)

param_dist = {
    "estimator__iterations": randint(200, 1200),
    "estimator__learning_rate": uniform(0.01, 0.29),
    "estimator__depth": randint(3, 11),
    "estimator__l2_leaf_reg": uniform(1.0, 9.0),
    "estimator__rsm": uniform(0.6, 0.4),
    "estimator__random_strength": uniform(0.0, 2.0),
    "estimator__border_count": randint(32, 255),
    "estimator__subsample": uniform(0.6, 0.4),
}

cv = KFold(n_splits=10, shuffle=True, random_state=42)

random_search = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_dist,
    n_iter=100,
    scoring="r2",
    cv=cv,
    verbose=2,
    random_state=42,
    n_jobs=-1,
    error_score="raise"
)

random_search.fit(X_train, y_train)

print("Best Hyperparameters:", random_search.best_params_)
print("Best R2:", random_search.best_score_)


Fitting 10 folds for each of 100 candidates, totalling 1000 fits
Best Hyperparameters: {'estimator__border_count': 43, 'estimator__depth': 4, 'estimator__iterations': 935, 'estimator__l2_leaf_reg': 1.5207750798193667, 'estimator__learning_rate': 0.12441132159635765, 'estimator__random_strength': 0.21351213187176876, 'estimator__rsm': 0.7342640259187853, 'estimator__subsample': 0.6678719615582261}
Best R2: 0.9705539597494237
